# Bénin Insights Challenge 2026 — Analyse Exploratoire GDELT

**iSHEERO × DataCamp Donates — Équipe 7**

Ce notebook analyse les données GDELT du Bénin sur l'année 2025 (janvier–décembre).  
Il répond aux 5 questions analytiques du projet et produit les insights destinés aux décideurs, journalistes et chercheurs béninois.

---

| Question | Sujet |
|---|---|
| **Q1** | Quand le monde parle-t-il du Bénin ? |
| **Q2** | Le ton médiatique est-il positif, neutre ou négatif ? |
| **Q3** | Combien de temps faut-il pour qu'un événement béninois soit couvert mondialement ? |
| **Q4** | Les sources médiatiques changent-elles pendant les périodes de crise ? |
| **Q5** | Le Bénin est-il acteur ou spectateur sur la scène internationale ? |

---
**Données** : GDELT v2 — BigQuery   
**Période** : Janvier 2025 → Décembre 2025  
**Pipeline** : 


## 0. Configuration et chargement des données

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Pipeline config
from pipeline.config import PROCESSED_DIR, SAMPLES_DIR, PIPELINE_VERSION

print(f"Pipeline version : {PIPELINE_VERSION}")
print(f"Pandas           : {pd.__version__}")
print(f"NumPy            : {np.__version__}")


In [ ]:
# ── Load data ──────────────────────────────────────────────────
# Priority: processed (full run) → sample (test run)

processed_file = PROCESSED_DIR / "benin_gdelt_clean.csv"
sample_file    = SAMPLES_DIR   / "benin_gdelt_sample.csv"

if processed_file.exists():
    df = pd.read_csv(processed_file, low_memory=False)
    data_source = "FULL (processed)"
elif sample_file.exists():
    df = pd.read_csv(sample_file, low_memory=False)
    data_source = "SAMPLE (5,000 rows — run pipeline in full mode for complete results)"
else:
    raise FileNotFoundError(
        "No data file found. Run the pipeline first:
"
        "  python -m pipeline.run_pipeline --mode sample"
    )

print(f"Source       : {data_source}")
print(f"Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")


In [ ]:
# ── Type restoration after CSV round-trip ──────────────────────
date_cols = ["SQLDATE", "DATEADDED", "event_date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Restore event_month as integer
if "event_month" in df.columns:
    df["event_month"] = pd.to_numeric(df["event_month"], errors="coerce").astype("Int64")

MONTH_LABELS = {
    1:"Jan", 2:"Fév", 3:"Mar", 4:"Avr", 5:"Mai", 6:"Jun",
    7:"Jul", 8:"Aoû", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Déc"
}

print("[OK] Types restored")
print(f"Date range: {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")


## 1. Vue d'ensemble du jeu de données

In [ ]:
# Dataset overview — completeness and key metrics
summary = pd.DataFrame({
    "Colonne": df.columns,
    "Type": df.dtypes.values.astype(str),
    "Valeurs manquantes": df.isna().sum().values,
    "% manquant": (df.isna().sum() / len(df) * 100).round(1).values,
    "Valeurs uniques": df.nunique().values
})
print(f"{'='*60}")
print(f"APERCU DU JEU DE DONNÉES")
print(f"{'='*60}")
print(f"Lignes         : {len(df):,}")
print(f"Colonnes       : {len(df.columns)}")
print(f"Période couverte: {df['SQLDATE'].min().strftime('%d %b %Y')} → {df['SQLDATE'].max().strftime('%d %b %Y')}")
print(f"Articles totaux : {df['NumArticles'].sum():,}")
print(f"Sources uniques : {df['source_domain'].nunique()}")
print()
print(summary[["Colonne", "Type", "% manquant", "Valeurs uniques"]].to_string(index=False))


---

## Q1 — Quand le monde parle-t-il du Bénin ?

**Visualisation 1 : Volume mensuel de couverture médiatique**

Cette visualisation montre le nombre d'articles publiés chaque mois sur des événements liés au Bénin.  
Les pics permettent d'identifier les périodes où l'attention internationale est la plus forte.


In [ ]:
# ── Q1 — VISUALIZATION 1 : Monthly media volume ───────────────
monthly = (
    df.groupby("event_month", as_index=False)
    .agg(
        nb_articles=("NumArticles",  "sum"),
        nb_events=("SQLDATE",  "count"),
        avg_tone=("AvgTone",  "mean")
    )
    .sort_values("event_month")
)
monthly["month_label"] = monthly["event_month"].map(MONTH_LABELS)

fig1 = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Nombre d'articles publiés par mois (2025)",
        "Nombre d'événements enregistrés par mois (2025)"
    ),
    vertical_spacing=0.15,
    shared_xaxes=True
)

fig1.add_trace(
    go.Bar(
        x=monthly["month_label"],
        y=monthly["nb_articles"],
        name="Articles",
        marker_color="#1a56db",
        text=monthly["nb_articles"].apply(lambda v: f"{v:,}"),
        textposition="outside"
    ),
    row=1, col=1
)
fig1.add_trace(
    go.Bar(
        x=monthly["month_label"],
        y=monthly["nb_events"],
        name="Événements",
        marker_color="#7e3af2",
        text=monthly["nb_events"].apply(lambda v: f"{v:,}"),
        textposition="outside"
    ),
    row=2, col=1
)

fig1.update_layout(
    height=600,
    title_text="<b>Q1 — Couverture médiatique du Bénin en 2025</b>",
    title_font_size=18,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig1.update_yaxes(gridcolor="#f0f0f0")
fig1.show()


**Lecture du graphique** : Chaque barre représente le volume total d'articles publiés dans le monde sur des événements liés au Bénin durant ce mois. Un pic indique une période d'attention médiatique intense — souvent liée à un événement politique, économique ou sécuritaire majeur. Comparer la hauteur des barres avec les événements béninois connus de 2025 permet de valider la qualité des données GDELT.

In [ ]:
# ── Q1b — Top 10 event types ───────────────────────────────────
event_types = (
    df.groupby("event_root_label", as_index=False)
    .agg(count=("SQLDATE", "count"), articles=("NumArticles", "sum"))
    .sort_values("count", ascending=True)
    .tail(10)
)

fig1b = px.bar(
    event_types,
    x="count",
    y="event_root_label",
    orientation="h",
    title="<b>Top 10 — Types d'événements les plus fréquents (Bénin 2025)</b>",
    labels={"count": "Nombre d'événements", "event_root_label": ""},
    color="count",
    color_continuous_scale="Blues",
    text="count"
)
fig1b.update_traces(textposition="outside")
fig1b.update_layout(
    height=450,
    plot_bgcolor="white",
    coloraxis_showscale=False
)
fig1b.show()

print("
Insight Q1 :")
top = event_types.sort_values("count", ascending=False).iloc[0]
print(f"  Type d'événement le plus fréquent : {top['event_root_label']} ({top['count']:,} événements)")


---

## Q2 — Le ton médiatique est-il positif, neutre ou négatif ?

**Visualisation 2 : Évolution du ton médiatique moyen par mois**

Le score  de GDELT mesure le sentiment moyen des articles (de -100 très négatif à +100 très positif).  
Un ton négatif signale des couvertures liées à des crises, conflits ou problèmes. Un ton positif signale des développements favorables.


In [ ]:
# ── Q2 — VISUALIZATION 2 : Tone evolution over time ──────────
tone_monthly = (
    df.groupby("event_month", as_index=False)
    .agg(
        avg_tone=("AvgTone", "mean"),
        avg_goldstein=("GoldsteinScale", "mean"),
        pct_crisis=("is_crisis_period", "mean")
    )
)
tone_monthly["month_label"] = tone_monthly["event_month"].map(MONTH_LABELS)
tone_monthly["color"] = tone_monthly["avg_tone"].apply(
    lambda v: "#d63031" if v < -2 else ("#00b894" if v > 2 else "#fdcb6e")
)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Ton moyen mensuel (AvgTone GDELT)",
        "Distribution : Positif / Neutre / Négatif"
    )
)

# Left: line + bar tone over months
fig2.add_trace(
    go.Bar(
        x=tone_monthly["month_label"],
        y=tone_monthly["avg_tone"].round(2),
        marker_color=tone_monthly["color"],
        name="Ton moyen",
        text=tone_monthly["avg_tone"].round(1),
        textposition="outside"
    ),
    row=1, col=1
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
fig2.add_hline(y=2, line_dash="dot",  line_color="#00b894",
               annotation_text="Seuil positif", row=1, col=1)
fig2.add_hline(y=-2, line_dash="dot", line_color="#d63031",
               annotation_text="Seuil négatif", row=1, col=1)

# Right: pie distribution
tone_dist = df["tone_category"].value_counts().reset_index()
fig2.add_trace(
    go.Pie(
        labels=tone_dist["tone_category"],
        values=tone_dist["count"],
        marker_colors=["#d63031", "#fdcb6e", "#00b894"],
        hole=0.4,
        name="Distribution"
    ),
    row=1, col=2
)

fig2.update_layout(
    height=450,
    title_text="<b>Q2 — Ton médiatique global sur le Bénin (2025)</b>",
    title_font_size=18,
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig2.show()

tone_dist_dict = df["tone_category"].value_counts(normalize=True).mul(100).round(1).to_dict()
print("
Insight Q2 :")
for k, v in tone_dist_dict.items():
    print(f"  {k} : {v}%")


**Lecture du graphique** : Un ton moyen négatif persistant sur plusieurs mois consécutifs signale une période de tension ou de crise couverte internationalement. Le graphique circulaire donne la répartition globale sur toute l'année — si la majorité des articles ont un ton négatif, cela reflète une image internationale du Bénin dominée par des problèmes perçus comme des conflits, instabilités ou crises humanitaires.

---

## Q3 — Combien de temps faut-il pour qu'un événement béninois soit couvert mondialement ?

**Visualisation 3 : Distribution du délai de propagation médiatique**

Le délai de propagation mesure le nombre de jours entre la date de l'événement () et la date d'indexation dans GDELT ().  
Un délai de 0 signifie une couverture immédiate. Un délai élevé signifie que l'événement a mis du temps à attirer l'attention internationale.


In [ ]:
# ── Q3 — VISUALIZATION 3 : Propagation delay distribution ────
if "propagation_delay_days" in df.columns:
    delay = df["propagation_delay_days"].dropna()
    delay = delay[delay >= 0]

    delay_stats = {
        "Médiane (jours)":  delay.median(),
        "Moyenne (jours)":  delay.mean().round(1),
        "90e percentile":   delay.quantile(0.90),
        "Couverture < 1j (%)":  (delay == 0).mean() * 100,
        "Couverture < 7j (%)":  (delay <= 7).mean() * 100,
    }

    fig3 = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Distribution du délai de propagation (jours)",
            "Délai médian par mois"
        )
    )

    fig3.add_trace(
        go.Histogram(
            x=delay.clip(upper=30),
            nbinsx=31,
            marker_color="#1a56db",
            name="Délai (jours)",
            opacity=0.8
        ),
        row=1, col=1
    )

    delay_monthly = (
        df.groupby("event_month")["propagation_delay_days"]
        .median()
        .reset_index()
    )
    delay_monthly["month_label"] = delay_monthly["event_month"].map(MONTH_LABELS)

    fig3.add_trace(
        go.Bar(
            x=delay_monthly["month_label"],
            y=delay_monthly["propagation_delay_days"].round(1),
            marker_color="#7e3af2",
            name="Délai médian",
            text=delay_monthly["propagation_delay_days"].round(1),
            textposition="outside"
        ),
        row=1, col=2
    )

    fig3.update_layout(
        height=420,
        title_text="<b>Q3 — Délai de propagation médiatique des événements béninois (2025)</b>",
        title_font_size=18,
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )
    fig3.update_xaxes(title_text="Délai (jours, max 30 affiché)", row=1, col=1)
    fig3.update_xaxes(title_text="Mois", row=1, col=2)
    fig3.update_yaxes(title_text="Nombre d'événements", row=1, col=1)
    fig3.update_yaxes(title_text="Délai médian (jours)", row=1, col=2)
    fig3.show()

    print("
Insight Q3 :")
    for k, v in delay_stats.items():
        print(f"  {k} : {v:.1f}")
else:
    print("[WARN] propagation_delay_days column not available in this dataset.")


**Lecture du graphique** : Si la majorité des barres se concentrent à 0-1 jour, GDELT détecte et indexe les événements béninois quasi immédiatement — ce qui valide la fiabilité de la base pour une veille en temps réel. Des pics à 7+ jours indiquent des événements qui n'ont attiré l'attention internationale que tardivement, souvent des crises locales qui ont dû s'aggraver avant d'être couvertes à l'étranger.

---

## Q4 — Les sources médiatiques changent-elles pendant les périodes de crise ?

**Visualisation 4 : Top sources en période normale vs période de crise**

Une crise est définie ici par un ton très négatif (AvgTone < -5) ou un score de destabilisation fort (GoldsteinScale < -5).  
Cette analyse révèle si les mêmes médias couvrent le Bénin en permanence, ou si des sources spécialisées apparaissent uniquement lors des crises.


In [ ]:
# ── Q4 — VISUALIZATION 4 : Sources crisis vs normal ──────────
crisis_df = df[df["is_crisis_period"] == True]
normal_df = df[df["is_crisis_period"] == False]

def top_sources(data, n=8):
    return (
        data["source_domain"]
        .value_counts()
        .head(n)
        .reset_index()
        .rename(columns={"source_domain": "Domaine", "count": "Occurrences"})
    )

top_crisis = top_sources(crisis_df)
top_normal = top_sources(normal_df)

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Période normale ({len(normal_df):,} événements)",
        f"Période de crise ({len(crisis_df):,} événements)"
    ),
    shared_xaxes=False
)

fig4.add_trace(
    go.Bar(
        x=top_normal["Occurrences"],
        y=top_normal["Domaine"],
        orientation="h",
        marker_color="#00b894",
        name="Normal",
        text=top_normal["Occurrences"],
        textposition="outside"
    ),
    row=1, col=1
)
fig4.add_trace(
    go.Bar(
        x=top_crisis["Occurrences"],
        y=top_crisis["Domaine"],
        orientation="h",
        marker_color="#d63031",
        name="Crise",
        text=top_crisis["Occurrences"],
        textposition="outside"
    ),
    row=1, col=2
)

fig4.update_layout(
    height=430,
    title_text="<b>Q4 — Sources médiatiques : périodes normales vs périodes de crise (2025)</b>",
    title_font_size=18,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig4.show()

# Identify sources that appear ONLY in crisis
crisis_only = set(top_crisis["Domaine"]) - set(top_normal["Domaine"])
print("
Insight Q4 :")
print(f"  Sources présentes uniquement en période de crise : {crisis_only if crisis_only else 'Aucune — mêmes sources en permanence'}")
pct_crisis = len(crisis_df) / len(df) * 100
print(f"  Proportion d'événements en période de crise : {pct_crisis:.1f}%")
print(f"  Nombre de sources uniques en crise    : {crisis_df['source_domain'].nunique()}")
print(f"  Nombre de sources uniques hors crise  : {normal_df['source_domain'].nunique()}")


**Lecture du graphique** : Si les mêmes domaines apparaissent dans les deux colonnes, la couverture du Bénin est assurée par un corpus de sources stable quel que soit le contexte. Si des domaines apparaissent uniquement à droite (période de crise), cela indique des médias spécialisés dans la couverture des conflits ou des urgences en Afrique de l'Ouest — information précieuse pour les décideurs qui veulent savoir qui les observe en cas de difficultés.

---

## Q5 — Le Bénin est-il acteur ou spectateur sur la scène internationale ?

**Visualisation 5 : Rôle du Bénin dans les événements GDELT**

Chaque événement GDELT implique un acteur 1 (initiateur) et un acteur 2 (cible ou partenaire).  
Cette analyse détermine si le Bénin initie les événements (**Acteur**), en est la cible (**Spectateur**), est les deux à la fois (**Mixte**), ou sert uniquement de cadre géographique (**Contexte**).


In [ ]:
# ── Q5 — VISUALIZATION 5 : Benin role distribution ──────────
role_counts = df["benin_role"].value_counts().reset_index()
role_colors = {
    "Acteur": "#1a56db",
    "Spectateur": "#7e3af2",
    "Mixte": "#f59e0b",
    "Contexte": "#6b7280"
}

fig5 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=(
        "Répartition globale",
        "Rôle du Bénin par type d'événement"
    )
)

# Pie chart
fig5.add_trace(
    go.Pie(
        labels=role_counts["benin_role"],
        values=role_counts["count"],
        marker_colors=[role_colors.get(l, "#aaa") for l in role_counts["benin_role"]],
        hole=0.45,
        textinfo="label+percent",
        name="Rôle"
    ),
    row=1, col=1
)

# Stacked bar: role × event type
role_event = (
    df.groupby(["event_root_label", "benin_role"])
    .size()
    .reset_index(name="count")
)
top_events = df["event_root_label"].value_counts().head(8).index.tolist()
role_event_top = role_event[role_event["event_root_label"].isin(top_events)]

for role in ["Acteur", "Spectateur", "Mixte", "Contexte"]:
    subset = role_event_top[role_event_top["benin_role"] == role]
    fig5.add_trace(
        go.Bar(
            x=subset["event_root_label"],
            y=subset["count"],
            name=role,
            marker_color=role_colors.get(role, "#aaa")
        ),
        row=1, col=2
    )

fig5.update_layout(
    height=480,
    title_text="<b>Q5 — Rôle du Bénin dans les événements internationaux (2025)</b>",
    title_font_size=18,
    barmode="stack",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title="Rôle"
)
fig5.show()

print("
Insight Q5 :")
for _, row in role_counts.iterrows():
    pct = row["count"] / len(df) * 100
    print(f"  {row['benin_role']:<15} : {row['count']:>5,} événements ({pct:.1f}%)")


**Lecture du graphique** : Un Bénin majoritairement en position **Contexte** signifie que la plupart des événements se déroulent sur son territoire sans qu'il soit l'initiateur — image d'un pays surtout objet de l'attention internationale plutôt que force diplomatique. Un fort pourcentage **Acteur** indiquerait un Bénin actif dans les négociations régionales, l'aide internationale ou les initiatives diplomatiques.

In [ ]:
# ── Q5b — Who talks about Benin? Actor type breakdown ─────────
actor_types = (
    df[df["benin_role"].isin(["Acteur", "Mixte"])]
    ["actor1_type_label"]
    .value_counts()
    .head(10)
    .reset_index()
)

fig5b = px.bar(
    actor_types,
    x="count",
    y="actor1_type_label",
    orientation="h",
    title="<b>Q5b — Qui parle du Bénin ? Types d'acteurs initiateurs</b>",
    labels={"count": "Nombre d'événements", "actor1_type_label": ""},
    color="count",
    color_continuous_scale="Purples",
    text="count"
)
fig5b.update_traces(textposition="outside")
fig5b.update_layout(
    height=420,
    plot_bgcolor="white",
    coloraxis_showscale=False
)
fig5b.show()


---

## Modèle de Machine Learning — Prédiction du ton médiatique

**Objectif** : Prédire si la couverture médiatique d'un événement béninois sera **positive**, **neutre** ou **négative**,  
à partir des caractéristiques de l'événement (type, stabilité, nombre de sources, rôle du Bénin).

**Valeur pour les décideurs** : Identifier en amont quels types d'événements génèrent une couverture négative du Bénin — pour anticiper et mieux communiquer.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import joblib

# ── Feature preparation ────────────────────────────────────────
FEATURES = [
    "GoldsteinScale",
    "NumArticles",
    "NumMentions",
    "NumSources",
    "event_month",
    "IsRootEvent",
    "QuadClass",
]

# Encode categorical features
le_role  = LabelEncoder()
le_event = LabelEncoder()

df_ml = df[FEATURES + ["tone_category", "benin_role", "event_root_label"]].copy()
df_ml = df_ml.dropna(subset=FEATURES + ["tone_category"])
df_ml = df_ml[df_ml["tone_category"] != "Inconnu"]

df_ml["benin_role_enc"]       = le_role.fit_transform(df_ml["benin_role"].fillna("Contexte"))
df_ml["event_root_label_enc"] = le_event.fit_transform(df_ml["event_root_label"].fillna("Autre"))

FEATURE_COLS = FEATURES + ["benin_role_enc", "event_root_label_enc"]

X = df_ml[FEATURE_COLS]
y = df_ml["tone_category"]

print(f"Dataset ML : {len(X):,} samples")
print(f"Classes    : {y.value_counts().to_dict()}")
print(f"Features   : {FEATURE_COLS}")


In [ ]:
# ── Train / Test split ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Model training — Random Forest ────────────────────────────
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",  # Handle class imbalance
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ── Cross-validation ───────────────────────────────────────────
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1_weighted")
print(f"Cross-validation F1 (5-fold) : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ── Evaluation on test set ─────────────────────────────────────
y_pred = model.predict(X_test)
print()
print("Classification Report — Test set:")
print(classification_report(y_test, y_pred))


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matrice de confusion — Random Forest", fontweight="bold")

# ── Feature importance ─────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
importances.plot(kind="barh", ax=axes[1], color="#1a56db", alpha=0.8)
axes[1].set_title("Importance des variables prédictives", fontweight="bold")
axes[1].set_xlabel("Importance (Gini)")
axes[1].axvline(x=importances.mean(), color="red", linestyle="--",
                label=f"Moyenne ({importances.mean():.3f})")
axes[1].legend()

plt.tight_layout()
plt.savefig("../models/confusion_matrix_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("[OK] Figure saved: models/confusion_matrix_feature_importance.png")


In [ ]:
# ── Save model for the ML Engineer ───────────────────────────
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/tone_classifier_rf.pkl")
joblib.dump(le_role, "../models/encoder_benin_role.pkl")
joblib.dump(le_event, "../models/encoder_event_root.pkl")
print("[OK] Model saved: models/tone_classifier_rf.pkl")
print("[OK] Encoders saved in models/")

# ── Interpretation ─────────────────────────────────────────────
top_feature = importances.index[-1]
top_importance = importances.values[-1]
print()
print("Interprétation :")
print(f"  Variable la plus prédictive : {top_feature} (importance = {top_importance:.3f})")
print(f"  Cela signifie que la stabilité politique et l'intensité médiatique")
print(f"  sont les principaux déterminants du ton de couverture du Bénin.")


---

## Synthèse — 5 Insights clés pour les décideurs

Ces insights sont issus directement de l'analyse ci-dessus. Ils sont rédigés en langage non technique.


In [ ]:
# ── Compute final insight numbers ─────────────────────────────
peak_month_idx = monthly.loc[monthly["nb_articles"].idxmax(), "event_month"]
peak_month_label = MONTH_LABELS.get(int(peak_month_idx), str(peak_month_idx))
peak_articles = monthly["nb_articles"].max()

tone_pct = df["tone_category"].value_counts(normalize=True).mul(100).round(1)
neg_pct  = tone_pct.get("Négatif", 0)
pos_pct  = tone_pct.get("Positif", 0)

med_delay  = df["propagation_delay_days"].median() if "propagation_delay_days" in df.columns else "N/A"
fast_pct   = (df["propagation_delay_days"] <= 1).mean() * 100 if "propagation_delay_days" in df.columns else 0

top_src      = df["source_domain"].value_counts().index[0] if len(df) > 0 else "N/A"
top_src_pct  = df["source_domain"].value_counts(normalize=True).iloc[0] * 100

role_pct = df["benin_role"].value_counts(normalize=True).mul(100).round(1)

print(f"""
INSIGHT 1 — PICS DE COUVERTURE
  Le mois de {peak_month_label} concentre le plus grand volume de couverture médiatique
  avec {peak_articles:,} articles publiés dans le monde. Identifier ce qui s'est passé
  ce mois-là permet de comprendre ce qui retient l'attention internationale sur le Bénin.

INSIGHT 2 — TON MAJORITAIREMENT NÉGATIF
  {neg_pct:.0f}% des articles sur le Bénin ont un ton négatif contre {pos_pct:.0f}% positif.
  Le Bénin est principalement couvert dans des contextes de tensions, conflits ou
  problèmes perçus — et non dans des contextes de réussite ou de développement.

INSIGHT 3 — COUVERTURE QUASI IMMÉDIATE
  Délai médian d'indexation : {med_delay} jour(s). {fast_pct:.0f}% des événements
  sont couverts en moins de 24 heures. GDELT est donc utilisable pour une veille
  en temps réel des événements béninois à l'échelle mondiale.

INSIGHT 4 — CONCENTRATION DES SOURCES
  {top_src} représente {top_src_pct:.1f}% de toute la couverture. La dépendance
  à un petit nombre de sources signifie que l'image internationale du Bénin
  est façonnée par quelques rédactions dont les biais éditoriaux comptent.

INSIGHT 5 — BÉNIN MAJORITAIREMENT EN POSITION DE CONTEXTE
  Dans {role_pct.get("Contexte", 0):.0f}% des événements, le Bénin est le cadre géographique
  mais pas l'initiateur. Le Bénin est acteur dans seulement {role_pct.get("Acteur", 0):.0f}%
  des cas — ce qui indique une couverture réactive plutôt qu'une diplomatie offensive.
""")
